# Book Recommender System

This notebook documents the development of the rating-based recommendation engine to support the user story: "As a user, I want to see a personalised "For You" feed of book cards so that I can discover books tailored to my preferences." 

The recommender uses a pipleine which implements two strategies:
1. If the user has fewer than 5 book ratings, popularity-based recommendations are produced using a Bayesian average score.
2. Otherwise collaborative filtering via Singular Value Decomposition (SVD) is used

Both strategies are served through `recommender_api.py` which is a Flask API that integrates with the Spring Boot backend, updating recommendations automatically as users rate books. The more the users rate, the more personalised the recommendations become.

## Imports

In [5]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sqlalchemy import create_engine
from surprise import SVD, Reader, Dataset
from surprise.model_selection import train_test_split as surprise_train_test_split
from surprise import accuracy
from surprise.model_selection import cross_validate
from sqlalchemy import text

## Loading Goodreads UCSD Book Graph Datasets

This dataset was chosen based on the intended target audience of this application - young adults who follow social media book communities (e.g. BookTok).

In [ ]:
#dataset has 93,398 books
books_ya = pd.read_json("../datasets/goodreads_books_young_adult.json.gz", lines=True)

#dataset has 34,919,254 interactions - interactions read will be truncated to simplify model
interactions = pd.read_json("../datasets/goodreads_interactions_young_adult.json.gz", lines=True, nrows=2000000)

books_ya.to_csv("../datasets/goodreads_books_young_adult.csv", index = False)
interactions.to_csv("../datasets/interactions_young_adult.csv", index = False)

In [ ]:
#loading author metadata
authors_df = pd.read_json( "../datasets/goodreads_book_authors.json.gz", lines=True)
authors_df.to_csv("../datasets/goodreads_book_authors.csv", index = False)

### Data preprocessing

The raw data requires several cleaning steps before it can be used for modelling...

In [ ]:
raw_df = interactions.merge(books_ya, on="book_id")

In [ ]:
#removing unnecessary columns

merged_df = raw_df[[
    "user_id",
    "book_id",
    "rating",
    "title",
    "authors",
    "description",
    "publication_year",
    "num_pages",
    "average_rating",
    "ratings_count",
    "image_url",
    "popular_shelves", # to extract the genre
    "language_code",
]].copy()

In [ ]:
print(f"Merged dataset shape: {merged_df.shape}")

### Merging author data with dataset

In [ ]:
#loading author data 
authors_df = pd.read_json("../datasets/goodreads_book_authors.json.gz", lines = True)
authors_df.to_csv("../datasets/goodreads_book_authors.csv", index = False)

In [ ]:
#converting both to string
authors_df["author_id"] = authors_df["author_id"].astype(str)

In [ ]:
#extracting author name
merged_df["author_id"] = merged_df["authors"].apply(
  lambda x: x[0]["author_id"] if isinstance(x, list) and len(x) > 0 else None
)

In [ ]:
authored_df = merged_df.merge(authors_df[["author_id","name"]], on="author_id", how="left")

In [ ]:
authored_df = authored_df.rename(columns={"name":"author"})

In [ ]:
#i only need author names 
authored_df = authored_df.drop(columns=["authors", "author_id"])

In [ ]:
authored_df[["title","author"]]

In [ ]:
#defining genre shelves to match against
GENRE_SHELVES = {
    "fantasy", "romance", "science-fiction", "sci-fi", "mystery", "thriller",
    "horror", "dystopia", "paranormal", "urban-fantasy", "historical-fiction",
    "contemporary", "adventure", "fiction", "non-fiction", "humor", "comedy",
    "paranormal-romance", "sci-fi-fantasy", "teen-fiction", "young-adult-fiction"
}

def extract_genre(shelves):
    if not isinstance(shelves, list):
        return "unknown"
    for shelf in shelves:
        if shelf["name"] in GENRE_SHELVES:
            return shelf["name"]
    return "unknown"

In [ ]:
authored_df["genre"] = authored_df["popular_shelves"].apply(extract_genre)
authored_df = authored_df.drop(columns=["popular_shelves"])

print(authored_df["genre"].value_counts())
print(f"\nUnknown: {(authored_df['genre'] == 'unknown').sum()}")

In [ ]:
# removing unrated interactions
clean_df = authored_df[authored_df["rating"] > 0].copy()

In [ ]:
print("After removing unrated:")
print("Users:", clean_df["user_id"].nunique())
print("Books:", clean_df["book_id"].nunique())

In [ ]:
# filtering books with almost no ratings
filtered_df = clean_df.groupby("book_id").filter(lambda x: len(x) >= 5)

In [ ]:
# filtering users who rated fewer than 5 books
filtered_df = filtered_df.groupby("user_id").filter(lambda x: len(x) >= 5) 

# filtering books without cover images
filtered_df = filtered_df[
    filtered_df["image_url"].notna() &
    (filtered_df["image_url"] != "") &
    (~filtered_df["image_url"].str.contains("nophoto", na = False))]

In [ ]:
print("\nFinal dataset:")
print("Shape:", filtered_df.shape)
print("Users:", filtered_df["user_id"].nunique())
print("Books:", filtered_df["book_id"].nunique())
print("\nRatings per user:")
print(filtered_df.groupby("user_id").size().describe())

## Exploratory Data Analysis

Before building models, the key properties of the dataset are examined:
- **Sparsity** — how much of the user-item matrix is filled
- **Rating distribution** — whether ratings are skewed
- **Interaction matrix** — the pivot table used as input to the SVD model

In [ ]:
n_users = filtered_df["user_id"].nunique()
n_books = filtered_df["book_id"].nunique()
n_ratings = len(filtered_df)
sparsity = 1 - (n_ratings / (n_users * n_books))

print(f"Users:    {n_users:,}")
print(f"Books:    {n_books:,}")
print(f"Ratings:  {n_ratings:,}")
print(f"Sparsity: {sparsity:.4f}")
print("\nNote: High sparsity is expected and normal for recommender systems.")
print("SVD handles sparse matrices well via latent factor decomposition.")

In [ ]:
#checking ratings are between 1-5
print("Rating distribution:")
print(filtered_df["rating"].value_counts().sort_index())

In [ ]:
#building user-item interaction matrix 
interaction_matrix = filtered_df.pivot_table(
    index = "user_id",
    columns = "book_id",
    values = "rating"
)

In [ ]:
print(f"Interaction matrix shape: {interaction_matrix.shape}")
interaction_matrix.head()

## Popularity-based recommender (cold start)

New users have no rating history, so collaborative filtering cannot be applied. Instead, they receive recommendations based on overall book popularity.

A **Bayesian average** is used rather than a simple average rating. This penalises books with very few ratings (a book with 3 ratings averaging 5.0 should not outrank a book with 10,000 ratings averaging 4.5).

The Bayesian average formula is:

$$score = \frac{n \cdot \bar{x} + C \cdot m}{n + C}$$

Where:
- $n$ = number of ratings for the book
- $\bar{x}$ = average rating for the book  
- $C$ = mean number of ratings across all books
- $m$ = global mean rating across all books

In [ ]:
#count ratings per book

num_rating_df = filtered_df.groupby("book_id")["rating"].count().reset_index()
num_rating_df.rename(columns= {"rating":"num_ratings"}, inplace = True)
num_rating_df

In [ ]:
#finding the average rating for each book

avg_rating_df = filtered_df.groupby("book_id")["rating"].mean().reset_index()
avg_rating_df.rename(columns= {"rating": "avg_rating"}, inplace = True)
avg_rating_df

In [ ]:
#merging the two tables to create a new popular books dataframe

book_info = filtered_df[["book_id", "title", "author","genre", "description","image_url"]].drop_duplicates("book_id")
popular_df = num_rating_df.merge(avg_rating_df, on = "book_id").merge(book_info, on="book_id")

In [ ]:
# measuring the Bayesian average score -- this means books with a higher volume of ratings are prioritised 
# over smaller volume of ratings but high ratings

C = popular_df["num_ratings"].mean() # average number of ratings across all books
m = popular_df["avg_rating"].mean() # global mean rating

popular_df["score"] = ((popular_df["num_ratings"] * popular_df["avg_rating"]) + (C*m)) / (popular_df["num_ratings"] + C)

In [ ]:
# filter to books with a minimum number of ratings then rank

MIN_RATINGS = 50

popular_df = popular_df[popular_df["num_ratings"] >= MIN_RATINGS]
popular_df = popular_df.sort_values("score", ascending = False).reset_index(drop=True)
popular_df

In [ ]:
#deduplicate by title (some books appear under multiple editions)

popular_df_no_duplicates = (popular_df
    .sort_values("num_ratings", ascending=False)
    .drop_duplicates(subset="title", keep="first")
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
rint(f"Popular books available for cold-start: {len(popular_df_no_duplicates)}")
print()
print(popular_df_no_duplicates.head(10)[["title", "author", "num_ratings", "avg_rating", "score"]])

## Matrix Factorisation using SVD (personalised recommendations based on users with existing ratings)

Once a user has rated 5 or more books, the system switches to **SVD-based collaborative filtering** using the [Surprise](https://surpriselib.com/) library.

SVD (Singular Value Decomposition) is a matrix factorisation technique that decomposes the user-item rating matrix into latent factors representing hidden preferences. It can then predict how a user would rate books they haven't seen yet.

### Model Parameters (tuned via cross-validation)
| Parameter | Value | Description |
|---|---|---|
| `n_factors` | 50 | Number of latent factors |
| `n_epochs` | 30 | Number of SGD iterations |
| `lr_all` | 0.005 | Learning rate |
| `reg_all` | 0.02 | Regularisation term |

In [ ]:
# define a Reader object to specify the rating scale
reader = Reader(rating_scale = (1,5))

# load into Surprise dataset
data = Dataset.load_from_df(
    filtered_df[["user_id", "book_id", "rating"]],
    reader
)

In [ ]:
# splitting the data into train and test sets
trainset, testset = surprise_train_test_split(data, test_size=0.2)

In [ ]:
# instantiate thej SVD model
svd = SVD()

# train the model on the training set
svd.fit(trainset)

In [ ]:
# predict ratings for the test set
predictions = svd.test(testset)

# compute and print RMSE (root mean squared error) and MAE (mean absolute error)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

predictions are off by less than one star on average

In [ ]:
# cross-validate tuned SVD to confirm hyperparameter choices
svd_tuned = SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.02)
cross_validate(svd_tuned, data, measures=["RMSE", "MAE"], cv=5, verbose=True)

In [ ]:
def get_svd_recommendations(user_id, n=10):
    """
    Generate top-n SVD recommendations for a Goodreads user.
    Predicts ratings for all unrated books and returns the highest scored.
    """
    
    # get all books this user hasn't rated
    rated_books = filtered_df[filtered_df["user_id"] == user_id]["book_id"].values
    all_books = filtered_df["book_id"].unique()
    unrated_books = [b for b in all_books if b not in rated_books]

    # predict ratings for all unrated books
    predictions_list = [svd.predict(user_id, book_id) for book_id in unrated_books]

    # sort by estimated rating descending
    predictions_list.sort(key=lambda x: x.est, reverse=True)
    top_n = predictions_list[:n]

    # build results dataframe
    book_ids = [p.iid for p in top_n]
    scores = [round(p.est, 3) for p in top_n]

    book_info = filtered_df[["book_id", "title", "author", "genre", "description", "image_url"]].drop_duplicates("book_id")
    results = pd.DataFrame({"book_id": book_ids, "predicted_rating": scores})
    results = results.merge(book_info, on="book_id", how="left")
    results = results.drop_duplicates(subset="title", keep="first")

    return results

In [ ]:
svd_tuned.fit(trainset)

In [ ]:
sample_user = filtered_df["user_id"].iloc[0]
print(f"Recommendations for user: {sample_user}\n")
print(get_svd_recommendations(sample_user))

## Recommendation pipeline

The two strategies are combined into a single `get_recommendations` function that selects the appropriate method based on how many books the user has rated.

- **< 5 ratings** → cold start (popularity-based)
- **≥ 5 ratings** → warm start (SVD collaborative filtering)

The threshold of 5 is chosen as the minimum for SVD to have enough signal to produce meaningful personalised results.

In [ ]:
def get_recommendations(user_id, n=10):
    """
    Returns top-n book recommendations for a user.
    - Cold start (< 5 ratings): falls back to popularity-based recommendations.
    - Warm start (>= 5 ratings): uses SVD collaborative filtering.
    """
    user_ratings = filtered_df[filtered_df["user_id"] == user_id]

    if len(user_ratings) < 5:
        # Cold start — return top popular books the user hasn't rated
        rated_ids = set(user_ratings["book_id"].values)
        recs = (
            popular_df_no_duplicates[~popular_df_no_duplicates["book_id"].isin(rated_ids)]
            .head(n)[["book_id", "title", "author", "genre", "description", "image_url", "score"]]
            .rename(columns={"score": "predicted_rating"})
            .copy()
        )
        recs["method"] = "popularity"
    else:
        # Warm start — SVD
        recs = get_svd_recommendations(user_id, n)
        recs["method"] = "svd"

    recs["user_id"] = user_id
    return recs

In [ ]:
# test cold start (user with < 5 ratings)
print("COLD START:")
print(get_recommendations("cold_user"))

# test warm start (user with >= 5 ratings)
warm_user = filtered_df.groupby("user_id").filter(lambda x: len(x) >= 5)["user_id"].iloc[0]
print("\nWARM START (SVD):")
print(get_recommendations(warm_user))